# Buyer GPT B2 against B1: the same prompts, side by side

**Sell Anything, notebook 6.** B2 is the second version of the buyer model. It is trained the same way as B1
(notebook 4), but on more sales: the 100 old ones plus the 150 new sales of seeds 2200–2349, written under the
quality checks, in prompt format 5 (no budget on the card). This notebook only reads the results of the held-out
evaluation; it does not run the model, so it opens in seconds on any machine.

**How the two models are compared.** `tools/eval_buyer_b1.py` gives both models **exactly the same prompts**:
the same buyer card, the same story so far and the same seller answer. Each model writes one line (an opening, a
question, a reaction or a scene), sampled the way the game samples (temperature 0.7, top-k 50, a fixed seed for
every prompt), and the line is checked by the program rules every training sale passed. So any difference between
the two columns comes from the model.

⚠️ This is not two live sales. Every prompt is cut from a written sale, so the story before each line is the written
one, not what the model said earlier. That makes the comparison fair line by line, but it cannot show what happens
when a model's own mistake leads to the next one.

In [1]:
import hashlib
import json
import re
import sys
from collections import Counter
from pathlib import Path

ML = Path.cwd().parent
sys.path.insert(0, str(ML))
from sellanything_ml.buyers.cat_market import MAX_NEW

EVAL = ML / "runs/sft_b2/eval"
RUNS = {"old": (ML / "runs/sft_b1/eval_v2_no_budget", EVAL / "b2_val_old"),   # (B1, B2) for each validation set
        "new": (EVAL / "b1_val_new", EVAL / "b2_val_new")}


def report(stem):
    return json.load(open(f"{stem}.json"))["tasks"]


def samples(stem):
    """The example lines the evaluation keeps: the first 12 of each task, in the order of the validation file."""
    out, task, cur = [], None, None
    for line in open(f"{stem}_samples.md").read().splitlines():
        if line.startswith("## "):
            task = line[3:].strip()
        elif m := re.match(r"\*\*(\w+)\*\* \(sale (\d+)\) — (.*)", line):
            cur = {"task": task, "name": m[1], "seed": m[2], "story": m[3]}
            out.append(cur)
        elif cur is not None:
            for key in ("sale", "model", "checks"):
                if line.startswith(f"- {key}: "):
                    cur[key] = line[len(key) + 4:]
    return out


def pairs(which):
    """B1's and B2's line for every kept example; both runs read the same file in the same order."""
    a, b = (samples(stem) for stem in RUNS[which])
    for x, y in zip(a, b, strict=True):
        assert (x["task"], x["name"], x["seed"], x["story"]) == (y["task"], y["name"], y["seed"], y["story"])
    return list(zip(a, b))


def show(which, task):
    for k, (x, y) in enumerate([p for p in pairs(which) if p[0]["task"] == task], 1):
        story = re.sub(r"<\|\w+\|>", "", x["story"]).split(" / ")
        print(f"[{k}] {x['name']} (sale {x['seed']})")
        for part in story:
            print("    " + part)
        print("  sale:", x["sale"])
        for label, row in (("B1:  ", x), ("B2:  ", y)):
            print("  " + label, row["model"])
            print("        check:", row["checks"])
        print()

## 1. Training B2

Same recipe as B1: start from `pretrain_large` (71M parameters), 3 epochs, lr 3e-4, loss on the target only. It ran
on one Modal A100 (`tools/modal_sft_b2.py`), batch 16, 4,844 steps.

In [2]:
for name in ("sft_b1", "sft_b2"):
    rows = [json.loads(line) for line in open(ML / "runs" / name / "metrics.jsonl")]
    best = min(rows, key=lambda r: r["val_loss"])
    print(f"{name}: {len(rows)} checks, best val loss {best['val_loss']:.4f} at step {best['step']}, "
          f"last {rows[-1]['val_loss']:.4f} at step {rows[-1]['step']}")
    print("   ", " ".join(f"{r['step']}:{r['val_loss']:.3f}" for r in rows))

sft_b1: 10 checks, best val loss 0.9063 at step 3733, last 0.9063 at step 3733
    400:1.236 800:1.143 1200:1.072 1600:1.052 2000:0.976 2400:0.931 2800:0.938 3200:0.930 3600:0.908 3733:0.906
sft_b2: 10 checks, best val loss 1.3156 at step 3000, last 1.3222 at step 4844
    500:1.687 1000:1.555 1500:1.464 2000:1.424 2500:1.369 3000:1.316 3500:1.343 4000:1.336 4500:1.324 4844:1.322


B1's validation loss was still falling at its last step (0.906). B2's lowest validation loss is 1.316 at step 3000, about 1.9 epochs in; after that it goes up to 1.343 and ends at 1.322, so the second half of training did not help and `best.pt` is the step 3000 weights. The two numbers cannot be compared with each other: B2's validation set adds 17 new sales, whose lines are longer and use many more different words, so they are simply harder to predict. That is why the rest of this notebook scores both models on the same prompts instead.

## 2. The two validation sets

Both models are scored on sales that **neither** trained on. The old set is the 10 validation sales B1 was already
scored on; the new set is 17 sales from the new batch (seeds 2200–2349) that were held out of B2's training.

In [3]:
train_b2 = {json.loads(line)["seed"] for line in open(ML / "data/processed/buyers_b2/train.jsonl")}
train_b1 = {json.loads(line)["seed"] for line in open(ML / "data/processed/buyers_b1/train.jsonl")}
print(f"training sales: B1 {len(train_b1)}, B2 {len(train_b2)}")
for which in ("old", "new"):
    rows = [json.loads(line) for line in open(ML / f"runs/eval_sets_b2/val_{which}_format5.jsonl")]
    seeds = {r["seed"] for r in rows}
    print(f"{which} set: {len(seeds)} sales, {len(rows)} prompts {dict(Counter(r['task'] for r in rows))}, "
          f"seeds {min(seeds)}–{max(seeds)}, in B1 training {len(seeds & train_b1)}, in B2 training {len(seeds & train_b2)}")

training sales: B1 90, B2 223
old set: 10 sales, 1195 prompts {'open': 100, 'ask': 98, 'react': 908, 'imagine': 89}, seeds 2004–2067, in B1 training 0, in B2 training 0
new set: 17 sales, 1998 prompts {'open': 170, 'ask': 163, 'react': 1513, 'imagine': 152}, seeds 2204–2343, in B1 training 0, in B2 training 0


B2 trained on 223 sales, B1 on 90. Neither set shares a single sale with either model's training data, so every line below is written for a sale the model has never seen. The new set is bigger (1,998 prompts against 1,195), and nine in ten prompts in both sets are reactions, so the reaction scores rest on the most data.

## 3. Scores

Each number is the share of lines that pass a check, over every prompt of the set. "Same direction" means the model
moved its offer the same way (up, down, keep, walk away) as the buyer in the written sale; "mean offer error" is how
far the model's new offer is from the sale's, in dollars.

In [4]:
ROWS = [("open: readable", "open", "readable"), ("ask: readable", "ask", "readable"),
        ("ask: new in meaning", "ask", "new_in_meaning"), ("react: readable", "react", "readable"),
        ("react: passes the rules", "react", "rules_ok"), ("react: same direction as the sale", "react", "same_direction"),
        ("react: amount said = offer", "react", "said_equals_offer"), ("react: reasonable move", "react", "reasonable"),
        ("react: mean offer error ($)", "react", "mean_offer_error"),
        ("imagine: readable", "imagine", "readable"), ("imagine: passes the rules", "imagine", "rules_ok")]
for which in ("old", "new"):
    b1, b2 = (report(stem) for stem in RUNS[which])
    print(f"{which} set{'':30}{'B1':>8}{'B2':>8}")
    for label, task, key in ROWS:
        fmt = "{:>8.2f}" if key == "mean_offer_error" else "{:>8.1%}"
        print(f"  {label:36}" + fmt.format(b1[task][key]) + fmt.format(b2[task][key]))
    w1, w2 = b1["react"]["withdrawals"], b2["react"]["withdrawals"]
    print(f"  {'react: walk-aways (sale has ' + str(w1['sale']) + ')':36}{w1['model']:>8}{w2['model']:>8}")
    print()

old set                                    B1      B2
  open: readable                        100.0%  100.0%
  ask: readable                         100.0%  100.0%
  ask: new in meaning                    89.8%   90.8%
  react: readable                        99.6%  100.0%
  react: passes the rules                90.2%   90.4%
  react: same direction as the sale      78.8%   77.5%
  react: amount said = offer             99.9%  100.0%
  react: reasonable move                 99.9%   99.9%
  react: mean offer error ($)             2.52    2.58
  imagine: readable                     100.0%  100.0%
  imagine: passes the rules              70.8%   38.2%
  react: walk-aways (sale has 11)           14      10

new set                                    B1      B2
  open: readable                         97.6%   98.2%
  ask: readable                         100.0%  100.0%
  ask: new in meaning                    96.9%   98.2%
  react: readable                        99.7%  100.0%
  react: pa

**Old set (the style B1 was trained on):** the two models are almost level. B2 is a little better at new questions (90.8% against 89.8%) and a little worse at moving the offer the same way as the sale (77.5% against 78.8%); none of these gaps is large.

**New set (the style of the 150 new sales):** B2 is clearly better at reactions. More reactions pass the rules (92.3% against 86.7%), more move the same way as the written buyer (74.2% against 67.4%), and the average gap to the sale's offer falls from $4.62 to $2.93. B1 walks away 39 times where the written buyers walked away 18 times; B2 walks away 14 times.

**Scenes are the exception, in both sets:** B2's scenes pass the rules far less often (38.2% against 70.8% on the old set, 11.9% against 59.9% on the new set).

## 4. Which rules the lines break

The evaluation counts the problems the program rules find (the five most common for each task).

In [5]:
for which in ("old", "new"):
    for task in ("react", "imagine"):
        for label, stem in zip(("B1", "B2"), RUNS[which]):
            probs = report(stem)[task].get("problems", {})
            print(f"{which} {task} {label} ({sum(probs.values())} counted):")
            for problem, n in probs.items():
                print(f"   {n:4}  {problem}")
        print()

old react B1 (89 counted):
     85  an unchanged offer must name what the seller just said
      1  sign - does not match the offer going from $N to $N
      1  "exactly what i want" gives no reason; say what was just said and what it means for your own life
      1  sign + does not match the offer going from $N to $N
      1  the line says $N but the offer is $N
old react B2 (87 counted):
     83  an unchanged offer must name what the seller just said
      2  sign + does not match the offer going from $N to $N
      1  "exactly what i want" gives no reason; say what was just said and what it means for your own life
      1  sign - does not match the offer going from $N to $N

old imagine B1 (10 counted):
      3  the line never names stairs or says what it is for
      2  the line never names laptop or says what it is for
      2  the line never names crown or says what it is for
      2  the line mentions stairs without picking it; name only the things picked
      1  the line menti

Almost every reaction that fails breaks the same rule: the offer stays the same but the line does not name anything the seller just said. On the new set B2 makes this mistake 114 times against B1's 199. For scenes, nearly every failure is a thing that was picked but never used in the line ("the line never names ..."), and B2's list is longer.

## 5. Every kept line, side by side: the old validation set

The evaluation keeps the first 12 prompts of each task. For each one: the end of the story the buyer sees, what the
buyer in the written sale said, then B1's and B2's line and what the program check says about it.

### Openings

In [6]:
show("old", "open")

[1] Leo (sale 2004)
    (nothing yet)
  sale: I've got plenty of free time to keep a cat entertained. | $50
  B1:   I'm retired and home all day, so I want a cat that likes company. | $70
        check: {"readable": true, "offer": 70, "spoken_ok": true}
  B2:   I'm retired, so I need a cat that likes company and meets my calm needs. | $60
        check: {"readable": true, "offer": 60, "spoken_ok": true}

[2] Ruby (sale 2004)
    (nothing yet)
  sale: I've missed having a cat around since mine passed away last year. | $60
  B1:   I miss my old cat and I want a cat to keep me company. | $70
        check: {"readable": true, "offer": 70, "spoken_ok": true}
  B2:   I lost my old cat last year, and I'm looking for someone to keep me company. | $100
        check: {"readable": true, "offer": 100, "spoken_ok": true}

[3] Mei (sale 2004)
    (nothing yet)
  sale: I live on a busy road, so a cat that's not scared of noise would be perfect. | $55
  B1:   I live on a busy road, so I need a cat th

Both models write a readable opening for all 12 buyers. B1's offers stay closer to the written sale here: the average gap is about $8 for B1 and about $16 for B2, mostly because of B2's large swings (Ruby $100 where the sale has $60, Felix $150 where it has $100, Chloe $40 where it has $65). Some of B2's wording is already off: Aisha needs a cat that "won't scare me" and Sam wants a cat that is "patient with them and a lot of patience".

### Questions

In [7]:
show("old", "ask")

[1] Ben (sale 2004)
    Your opening ($75): I'm a bit of a perfectionist when it comes to my pets.
  sale: Can you tell me if the cat has any health issues and if all the vaccinations are up to date?
  B1:   Can you tell me more about the cat's health and if it has any known health issues?
        check: {"readable": true, "new_question": true, "new_in_meaning": true}
  B2:   Can you tell me more about the cat's health history and if it has had all its vaccinations up to date?
        check: {"readable": true, "new_question": true, "new_in_meaning": true}

[2] Leo (sale 2004)
    Ben asked: Can you tell me if the cat has any health issues and if all the vaccinations are up to date?
    Seller: Yea, shes FIV pos but shes fully vaccinated and had a vet check last month. She's all up to date on her stuff.
    You ($50 -> $55): Being FIV positive means she stays indoors, which suits me since I'm home all day, and a vet check last month is reassuring. I'll go up to $55.
  sale: Can you tell

Every question is readable and new, but B1 asks two that repeat an earlier question in meaning ([5] FIV and other cats, [9] how active she is), and one with a broken word ([8] "heroving nature"). B2's questions repeat less but some ramble or lose their meaning: [2] "well socialized for long years with her", [10] "was it ever my two?", [12] "her grace and if she's been allowed to go in a lot?".

### Reactions

In [8]:
show("old", "react")

[1] Leo (sale 2004)
    Your opening ($50): I've got plenty of free time to keep a cat entertained.
    Ben asked: Can you tell me if the cat has any health issues and if all the vaccinations are up to date?
    Seller: Yea, shes FIV pos but shes fully vaccinated and had a vet check last month. She's all up to date on her stuff.
  sale: + | Being FIV positive means she stays indoors, which suits me since I'm home all day, and a vet check last month is reassuring. I'll go up to $55. | $55
  B1:   0 | A vet check last month is something I have time for. Staying at $50. | $50
        check: {"readable": true, "rules_ok": true, "same_direction": false, "withdrew": false, "teacher_withdrew": false, "offer_error": 5, "said_equals_offer": true, "reasonable": true}
  B2:   0 | A vet check last month is good, but I'll hear whether she likes routine care. I'll stay at $50. | $50
        check: {"readable": true, "rules_ok": true, "same_direction": false, "withdrew": false, "teacher_withdrew": fa

This set shows the same weakness in both models: they misread the seller. The seller says the cat is **fully vaccinated** and had a vet check last month, and still B1 ([4], [5], [7]) and B2 ([4], [5]) talk about "no vaccinations". Two of B2's mistakes are worse than B1's: [4] Felix, who breeds show cats, lowers to $90 instead of walking away from an FIV-positive cat, and [5] Ben raises his offer while saying the cat is "the opposite of the perfect health I want". B2 also fails the rules twice ([11] "exactly what I want", [12] an unchanged offer that names nothing the seller said); B1 passes all 12 but [9] Chloe's line is broken English. So on this set B2's reactions are not better, which matches the scores.

### Scenes

In [9]:
show("old", "imagine")

[1] Leo (sale 2004)
    Owen asked: Can you tell me how she handles being in a small space and if she's likely to be inactive, since I live in a small apartment?
    Seller: She's pretty laid back and doesn't mind small spaces, so she should do fine in an apartment. She's not super active and likes her quiet time, so she won't take up too much space or cause a ruckus.
    You ($52 -> $52): Not super active fits my calm days at home. Staying at $52.
  sale: chair, radio, clock | She'd keep me company by my chair while the radio plays my old songs, and the clock tells us when it's time for her dinner.
  B1:   chair, teapot, clock | She'd nap by my chair while I pour from my teapot, and I'd set my alarm clock for her breakfast on my lap time.
        check: {"readable": true, "rules_ok": false} — the line mentions alarm clock without picking it; name only the things picked
  B2:   chair, door, pillow | She'd sit on my lap while I read my book and pill when I'm home, and I'd wear my old pi

B1 passes 7 of these 12 scenes, B2 only 3. B1's failures are mostly one thing left unused ([2] the teapot, [8] the alarm clock). B2 often leaves two of its three things unused ([1] chair and door, [8] broom and wristwatch, [9] house plant and alarm clock, [10] sink and door), and its sentences break more: [1] "read my book and pill ... I'd wear my old pillow to sleep until my morning pillow", [8] "I's hardly home". Even B2's passing scene [5] picks the alarm clock twice and says "so her breakfast long before I wake".

## 6. Every kept line, side by side: the new validation set

The same 48 kinds of prompt, from the 17 new held-out sales.

### Openings

In [10]:
show("new", "open")

[1] Chloe (sale 2204)
    (nothing yet)
  sale: Hi! I'm paying rent on a shared flat out of a part time café job, so I'm hoping for a kitten that won't break me. | $55
  B1:   I'm a student on a tight budget, so I'm looking for a good deal. | $60
        check: {"readable": true, "offer": 60, "spoken_ok": true}
  B2:   I'm the one on a budget, so I know what a cat needs to do, but I'd love that bills. | $55
        check: {"readable": true, "offer": 55, "spoken_ok": true}

[2] Jonah (sale 2204)
    (nothing yet)
  sale: Four lanes of traffic run right past my front step, day and night. | $72
  B1:   Hi. I'm on a busy road. | $55
        check: {"readable": true, "offer": 55, "spoken_ok": true}
  B2:   Lorries rumble past my front step on Sorrel Parade from dawn to midnight, so whoever comes home with me has to stay in. | $74
        check: {"readable": true, "offer": 74, "spoken_ok": true}

[3] Hana (sale 2204)
    (nothing yet)
  sale: Everything in my place is cream, brass and marble

Here the picture turns around. B2's openings sound like the written sales: concrete and tied to the card (Jonah: "Lorries rumble past my front step ... whoever comes home with me has to stay in"; Leo: "the house ... has been far too quiet since"), while B1's are short and generic ("Hi. I'm on a busy road.", "Student here, small budget and small budget."). B2's offers are much closer to the sale: about $5 away on average against about $17 for B1, which opens too low (Owen $70 where the breeder in the sale offers $110). But B2's longer sentences break more often: [6] "attoo is two", [3] "full of sunny grease and brass", [1] "I'd love that bills", [7] "I can't help a single room with my neighbour's books".

### Questions

In [11]:
show("new", "ask")

[1] Owen (sale 2204)
    Your opening ($110): My cattery has produced four grand champions, and every kitten I add has to strengthen a bloodline.
  sale: Is she a pedigree, and does she come with any registration papers?
  B1:   Is the cat the pedigree cat and does it have any specific breed traits that would make her suitable for breeding?
        check: {"readable": true, "new_question": true, "new_in_meaning": true}
  B2:   Before anything else, is she still intact, and is she still entire for breeding?
        check: {"readable": true, "new_question": true, "new_in_meaning": true}

[2] Jonah (sale 2204)
    Owen asked: Is she a pedigree, and does she come with any registration papers?
    Seller: She's a two month old black domestic shorthair, and that coat shines like wet ink. No papers that I know of, but she carries herself like a champion already.
    You ($72 -> $72): Papers or not, what I need to know is whether she'll keep clear of the road outside my door. Holding at $72.
 

B2's questions follow the asker's card more often: the breeder asks whether she is still entire [1], the parent asks about small children [4], the allergic buyer about fur [8], the hiker about a harness [12]. B1 has several broken questions: [4] "Is the kitten shots being a monthly ...", [11] "a purebred Bush", and [3] turns a question into a walk-away ("so I'm out. | out"). B2 still slips: [6] asks about going outdoors, which was already answered, [9] the buyer with a new couch asks about other cats instead of scratching, and [11] calls the cat "he".

### Reactions

In [12]:
show("new", "react")

[1] Chloe (sale 2204)
    Your opening ($55): Hi! I'm paying rent on a shared flat out of a part time café job, so I'm hoping for a kitten that won't break me.
    Owen asked: Is she a pedigree, and does she come with any registration papers?
    Seller: She's a two month old black domestic shorthair, and that coat shines like wet ink. No papers that I know of, but she carries herself like a champion already.
  sale: 0 | No papers keeps her at a market price, which is the only kind of price my café wages can manage. Holding at $55. | $55
  B1:   0 | Papers don't matter to me. $55. | $55
        check: {"readable": true, "rules_ok": true, "same_direction": true, "withdrew": false, "teacher_withdrew": false, "offer_error": 0, "said_equals_offer": true, "reasonable": true}
  B2:   0 | Papers never cost me a cent, and a breed name won't make my rent any cheaper to keep. Staying at $55. | $55
        check: {"readable": true, "rules_ok": true, "same_direction": true, "withdrew": false, "tea

B2's reasons tie the answer to the buyer's own life more often, as the new sales teach: Chloe [1] "Papers never cost me a cent, and a breed name won't make my rent any cheaper" against B1's "Papers don't matter to me. $55." Both models miss the same three raises, where the written buyers saw a plus in a two-month-old kitten ([4] learning odd hours, [9] riding in a pack, [10] sixteen more years). Both get [12] wrong: the cat has been indoor-outdoor, which is bad news on a busy road, yet both raise, and B2 even reads it as "she stays safe inside". B2 also invents or contradicts facts: [6] "one month old" (the seller said two) and [2] "the traffic was never my worry" from a buyer whose card is the traffic. [11] fails the rules.

### Scenes

In [13]:
show("new", "imagine")

[1] Chloe (sale 2204)
    Hana asked: Does she have any funny little habits I should know about before taking her home?
    Seller: Just one, and it's adorable: she'll only drink from a running tap. Honestly, watching her lap at the stream is like a little photo shoot every time.
    You ($41 -> $35): A tap left running for her drinks would push up the water bill I split with three flatmates. Down to $35.
  sale: laptop, book, coffee cup | I'd study at my laptop with a secondhand book open while she naps nearby, and my coffee cup keeps me awake through her noisy evenings.
  B1:   book, backpack, calculator | She'd nap on my book while I study with my book, I'd tally her spay on my calculator, and carry it home in my backpack.
        check: {"readable": true, "rules_ok": true}
  B2:   broom, spreadsheet, book | I'd clean her with my broom every morning after a café shift, then spreadsheet her vet costs on my spreadsheet.
        check: {"readable": true, "rules_ok": false} — the line n

On the new set B1 passes 8 of the 12 kept scenes and B2 only 1. B2's scenes often have nothing to do with the things it picked: [2] scissors, door and flashlight become "cut paper cloth ... so she can sit on the road" and a microphone; [5] map, shoe and school bus become a sentence about a salsa shoe; [12] it picks "bike", which is not on the drawable list. Words break apart: "theuss", "the walnut pillow I built", "I'd hang my bike in the hall after every bike". [10] even ends without a full stop, which may be the length limit cutting it off. The written scenes in this set are long and personal (Elena's [5] is 39 words), and B2 seems to imitate that length without being able to keep track of the three things.

## 7. Why are B2's scenes worse?

Three kinds of mistake show up in the scenes above: **(1)** a thing is picked but never used in the line, **(2)** the
line names a thing that was not picked or is not on the drawable list, **(3)** the sentence itself breaks. The first
two are counted by the program check; the third has to be read. First the counts over the 24 kept scenes of each
model:

In [14]:
def kind(row):
    problem = row["checks"].split(" — ")[-1] if " — " in row["checks"] else ""
    picked = [t.strip() for t in row["model"].split("|")[0].split(",")]
    return {"(1) picked, not used": "never names" in problem,
            "(2) not picked / not drawable": "without picking" in problem or "not on the list" in problem,
            "same thing picked twice": len(picked) != len(set(picked)),
            "passes": not problem}

for label, col in (("B1", 0), ("B2", 1)):
    total = Counter()
    for which in ("old", "new"):
        for p in pairs(which):
            if p[0]["task"] == "imagine":
                total.update(k for k, v in kind(p[col]).items() if v)
    print(label, dict(total))

B1 {'(2) not picked / not drawable': 2, '(1) picked, not used': 7, 'passes': 15, 'same thing picked twice': 2}
B2 {'(1) picked, not used': 15, '(2) not picked / not drawable': 5, 'passes': 4, 'same thing picked twice': 4}


Over the 24 kept scenes per model, B1 passes 15 and B2 only 4. B2 leaves picked things unused in 15 scenes (B1 in 7), names unpicked or undrawable things in 5 (B1 in 2), and picks the same thing twice in 4 (B1 in 2). The third kind, broken sentences, is not counted by any rule, but the lines above show it in B2 far more often.

One possible cause is the length limit: in the game and in the evaluation a scene may use at most `MAX_NEW["imagine"]`
tokens, and a line cut short would leave its last thing unused. How long are the written scenes?

In [15]:
import sentencepiece as spm
sp = spm.SentencePieceProcessor(model_file=str(ML / "checkpoints/tokenizer/sp8192.model"))
print("limit for a scene:", MAX_NEW["imagine"], "tokens")
for label, path in (("B1 training", "data/processed/buyers_b1/train.jsonl"),
                    ("B2 training", "data/processed/buyers_b2/train.jsonl"),
                    ("old set", "runs/eval_sets_b2/val_old_format5.jsonl"),
                    ("new set", "runs/eval_sets_b2/val_new_format5.jsonl")):
    n = sorted(len(sp.encode(json.loads(line)["target"])) for line in open(ML / path)
               if json.loads(line)["task"] == "imagine")
    over = sum(x > MAX_NEW["imagine"] for x in n)
    print(f"{label:12} {len(n):5} scenes, median {n[len(n) // 2]} tokens, longest {n[-1]}, "
          f"over the limit {over} ({over / len(n):.0%})")

limit for a scene: 64 tokens
B1 training    762 scenes, median 40 tokens, longest 63, over the limit 0 (0%)


B2 training   1987 scenes, median 50 tokens, longest 85, over the limit 111 (6%)
old set         89 scenes, median 41 tokens, longest 58, over the limit 0 (0%)
new set        152 scenes, median 56 tokens, longest 76, over the limit 17 (11%)


B1 never saw a scene longer than the 64-token limit; 6% of B2's training scenes and 11% of the new set's are longer. So some of B2's scenes are cut before they finish, which leaves the last thing unused (the cut-off line [10] of the new set may be one of them). But most of B2's failed scenes above are finished sentences that simply do not use their things, so the limit explains only a small part.

Another difference between the old and the new sales is which things their scenes use. How many different things
does each group of training scenes pick, and how often is a thing picked that is used fewer than five times?

In [16]:
def things(path, keep):
    count, scenes = Counter(), 0
    for line in open(ML / path):
        r = json.loads(line)
        if r["task"] == "imagine" and keep(r["seed"]):
            scenes += 1
            count.update(t.strip() for t in r["target"].split("|")[0].split(","))
    return count, scenes

for label, path, keep in (("B1 training", "data/processed/buyers_b1/train.jsonl", lambda s: True),
                          ("B2 training, old sales", "data/processed/buyers_b2/train.jsonl", lambda s: s < 2200),
                          ("B2 training, new sales", "data/processed/buyers_b2/train.jsonl", lambda s: s >= 2200)):
    count, scenes = things(path, keep)
    rare = sum(n for n in count.values() if n < 5) / sum(count.values())
    print(f"{label:24} {scenes:5} scenes, {len(count):3} different things, rare picks {rare:.0%}, "
          f"most used {count.most_common(4)}")

B1 training                762 scenes,  96 different things, rare picks 2%, most used [('pillow', 90), ('door', 89), ('picture frame', 72), ('alarm clock', 61)]


B2 training, old sales     762 scenes,  96 different things, rare picks 2%, most used [('pillow', 90), ('door', 89), ('picture frame', 72), ('alarm clock', 61)]
B2 training, new sales    1225 scenes, 231 different things, rare picks 5%, most used [('pencil', 75), ('dog', 72), ('scissors', 69), ('book', 68)]


The old sales picked from 96 different things, again and again (pillow 90 times, door 89). The new sales pick from 231 different things, and the most used one, pencil, appears only 75 times in 1,225 scenes. With more than twice as many things, each one is seen far less often, so the model has fewer examples of how to use any one of them. Together with the longer scenes, this is a likely reason why B2's scenes got worse; section 8 tests how much of it holds.

## 8. Experiments: what makes the scenes fail?

The numbers above suggest causes but do not test them. These runs do (`tools/probe_imagine.py`, on the CPU, every
imagine prompt of both sets, 89 + 152 = 241 scenes):

- **b1_64, b2_64**: each model picks its own things and writes the line, with the usual limit of 64 tokens.
- **b2_128**: the same, but B2 may write up to 128 tokens. If cut-off lines were the problem, this would fix it.
- **b1_forced, b2_forced**: the scene starts with the three things the written sale picked, and the model only
  writes the line. This separates *choosing* things from *writing* about them.

The CPU draws different random numbers from the Mac's GPU, so these are not the same samples as section 3; the runs
are compared with each other.

In [17]:
from sellanything_ml.buyers.stories import _named

PROBE = ML / "runs/sft_b2/imagine_probe"
probe = {name: [json.loads(line) for line in open(PROBE / f"{name}.jsonl")]
         for name in ("b1_64", "b2_64", "b2_128", "b1_forced", "b2_forced")}


def kind(r):
    p = r.get("problem") or ""
    if r.get("rules_ok"):
        return "passes"
    if not r.get("readable"):
        return "unreadable"
    if "never names" in p:
        return "picked, not used"
    if "without picking" in p:
        return "not picked"
    if "not on the list" in p or "cannot be drawn" in p:
        return "not drawable"
    return "other"


for name, rows in probe.items():
    rate = {s: sum(bool(r.get("rules_ok")) for r in rows if r["set"] == s) / sum(r["set"] == s for r in rows)
            for s in ("old", "new")}
    print(f"{name:10} old {rate['old']:4.0%}  new {rate['new']:4.0%}  cut off {sum(r['hit_limit'] for r in rows):3}  "
          f"{dict(Counter(kind(r) for r in rows))}")

b1_64      old  70%  new  68%  cut off   0  {'passes': 165, 'picked, not used': 65, 'not drawable': 2, 'not picked': 8, 'unreadable': 1}
b2_64      old  42%  new  20%  cut off  15  {'passes': 67, 'picked, not used': 157, 'not picked': 7, 'other': 4, 'not drawable': 6}
b2_128     old  42%  new  20%  cut off   0  {'passes': 68, 'picked, not used': 156, 'not picked': 7, 'other': 4, 'not drawable': 6}
b1_forced  old  44%  new   1%  cut off   0  {'passes': 41, 'picked, not used': 194, 'not picked': 5, 'unreadable': 1}
b2_forced  old  28%  new   9%  cut off   0  {'picked, not used': 200, 'passes': 39, 'not picked': 2}


**The length limit is not the cause.** With 128 tokens instead of 64, B2 passes 28.2% instead of 27.8%: of the 15 lines that were cut off at 64, only one more passes when it may finish.

**Choosing is where B1's scores come from.** Given the written sale's three things, B1 passes 44% of the old set and only **1%** of the new set. Its 68% on the new set (choosing its own things) comes from picking things it knows how to write about, not from being able to write about the things the new sales use. B2, given the same things, passes 28% of the old set and 9% of the new set: a little better than B1 on the new things, but far from usable.

Almost every failure in every run is the same kind: a picked thing that the line never uses.

Then, for every thing a model put in its list, was it used in the line? Grouped by how many training scenes of that
model picked the same thing, and by its place in the list:

In [18]:
seen = {}                                 # how many training scenes picked each thing
for label, path in (("b1", "data/processed/buyers_b1/train.jsonl"), ("b2", "data/processed/buyers_b2/train.jsonl")):
    seen[label] = Counter()
    for line in open(ML / path):
        r = json.loads(line)
        if r["task"] == "imagine":
            seen[label].update(t.strip() for t in r["target"].split("|")[0].split(","))

BUCKETS = ("never", "1-9", "10-29", "30+")
def bucket(n):
    return "never" if n == 0 else "1-9" if n < 10 else "10-29" if n < 30 else "30+"

for name, rows in probe.items():
    used, picked, unused_at = Counter(), Counter(), [0, 0, 0]
    for r in rows:
        if "|" not in r["model"]:
            continue
        things, line = r["model"].split("|", 1)
        things = [t.strip() for t in things.split(",")]
        if len(things) != 3:
            continue
        for k, t in enumerate(things):
            b = bucket(seen[name[:2]][t])
            picked[b] += 1
            if _named(t, line):
                used[b] += 1
            else:
                unused_at[k] += 1
    rates = "  ".join(f"{b}: {used[b]}/{picked[b]} {used[b] / picked[b]:.0%}" for b in BUCKETS if picked[b])
    print(f"{name:10} {rates}   unused 1st/2nd/3rd thing {unused_at}")

b1_64      never: 1/2 50%  1-9: 17/19 89%  10-29: 49/56 88%  30+: 570/643 89%   unused 1st/2nd/3rd thing [26, 25, 32]
b2_64      never: 2/6 33%  1-9: 6/8 75%  10-29: 60/94 64%  30+: 434/603 72%   unused 1st/2nd/3rd thing [76, 70, 63]
b2_128     never: 2/6 33%  1-9: 6/8 75%  10-29: 61/94 65%  30+: 434/603 72%   unused 1st/2nd/3rd thing [76, 69, 63]
b1_forced  never: 4/177 2%  1-9: 36/126 29%  10-29: 34/68 50%  30+: 271/352 77%   unused 1st/2nd/3rd thing [124, 131, 123]
b2_forced  never: 0/3 0%  1-9: 12/51 24%  10-29: 89/149 60%  30+: 344/520 66%   unused 1st/2nd/3rd thing [100, 93, 85]


**The rarer the thing, the less often it gets used.** When the things are given, B1 uses a thing it never saw in a training scene 2% of the time, one it saw 1–9 times 29%, and one it saw 30 times or more 77%; B2 uses 24% of the things seen 1–9 times and 66% of those seen 30 times or more. The new sales picked from 231 different things (section 7), many of them seen only a few times, so this explains a large part of the drop on the new set.

**But rarity is not the whole story.** When B2 picks its own things they are mostly common ones (603 of its 711 picks were seen 30 times or more), and it still uses only 72% of those, against B1's 89%. The unused thing is spread evenly over the first, second and third place, so it is not a line running out of room. Something in how B2 learned to write scenes also got worse; the longer, more personal scene lines of the new sales (29 words against 21) are a likely reason, but that is not tested here.

## 9. What this comparison says

1. **Reactions, the core of the game, are better in B2 on the new style of sale.** On the 17 new held-out sales B2 moves its offer the way the written buyer did more often (74.2% against 67.4%), lands closer to the written offer ($2.93 against $4.62 away), walks away a sensible number of times (14 against the sale's 18; B1 39), and ties its reasons to the buyer's life more often.
2. **On the old style of sale B2 is no better than B1.** Both misread the seller in the same places ("fully vaccinated" read as "no vaccinations"), and B2 makes two serious mistakes B1 does not (the breeder not walking away, a raise with a reason against it).
3. **B2's scenes are much worse, and the experiments narrow down why.** The 64-token limit is not the cause. Things a model has seen in few training scenes are rarely used, and the new sales use 231 different things, many of them rare. Even common things are used less by B2 (72% against 89%), which points at the longer scene lines as well, but that part is not tested.
4. **What could be changed in the data or the training** (none of it tried yet; each would need a new training run): pick scene things from a smaller set that every scene sees many times, or write more scenes for the rarer things; keep scene lines short, about one clause per thing, as in the old sales; give the scene examples more weight in training, since they are only 8% of the examples (1,987 of 25,836).
5. **What this comparison cannot show:** every line here starts from a written story, so it does not show how B2 does in a live sale, where its own earlier mistakes would carry forward. Playing the same cat as notebook 5 with B2 is the next check.

## Version info

The files this notebook read, so the numbers can be traced back.

In [19]:
for f in ["runs/sft_b2/best.pt", "data/processed/buyers_b2/train.jsonl", "data/processed/buyers_b2/val.jsonl",
          "runs/eval_sets_b2/val_old_format5.jsonl", "runs/eval_sets_b2/val_new_format5.jsonl",
          "runs/sft_b1/eval_v2_no_budget.json", "runs/sft_b2/eval/b2_val_old.json",
          "runs/sft_b2/eval/b1_val_new.json", "runs/sft_b2/eval/b2_val_new.json"]:
    print(hashlib.sha256((ML / f).read_bytes()).hexdigest()[:16], f)

a8df01c605a09522 runs/sft_b2/best.pt
6043ee1a47a79cc6 data/processed/buyers_b2/train.jsonl
16f7326c10afd42a data/processed/buyers_b2/val.jsonl
3e52ea5848e29038 runs/eval_sets_b2/val_old_format5.jsonl
c319c04a9beb8736 runs/eval_sets_b2/val_new_format5.jsonl
5729ae5334185c4f runs/sft_b1/eval_v2_no_budget.json
ece821a22caa0c02 runs/sft_b2/eval/b2_val_old.json
629d9024f070e3df runs/sft_b2/eval/b1_val_new.json
21716beb3fe3f8fc runs/sft_b2/eval/b2_val_new.json


The B2 data files were written at 16:01 on 2026-10-12, a minute before the Modal run uploaded them, so these hashes are the data B2 trained on. The B2 entry in `docs/model-versions.md` was written before training (it still says "not trained yet" and lists an older hash for train.jsonl), so it needs updating with these values.